TITRE

INTRODUCTION

SOMMAIRE

Installation

In [1]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
#Fonctions:
import src.clear_data as cd
import src.get_data as gd

Préparation des données

Adresses

In [2]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

Import et modifications/nettoyage des données

In [3]:
#Import
df_dvf_raw = gd.fetch_dvf_api()
gdf_metro_raw = gd.fetch_metro_api()
#Nettoyage
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)
gdf_final['prix_m2'] = gdf_final['valeur_fonciere'] / gdf_final['surface_reelle_bati']
print(gdf_final.head)
print(gdf_final.columns)

--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération des données Métro via API (Rennes Métropole) ---
<bound method NDFrame.head of         date_mutation nature_mutation  valeur_fonciere code_commune  \
1365203    2023-01-04           Vente         160000.0        35238   
1365206    2023-01-07           Vente         320000.0        35238   
1365214    2023-01-06           Vente         120000.0        35238   
1365215    2023-01-06           Vente         120000.0        35238   
1365243    2023-01-04           Vente         310000.0        35238   
...               ...             ...              ...          ...   
1427961    2023-12-20           Vente         180000.0        35238   
1428108    2023-12-22           Vente         151480.0        35238   
1428292    2023-11-17           Vente         263000.0        35238   
1428428    2023-11-14           Vente         270000.0        35238   
1428450    2023-08-30           Vente         250000.0        35238   

        nom_commune   type_local  surface_reelle_ba

Statistiques Descriptives 

In [6]:
print(gdf_final.head)

<bound method NDFrame.head of         date_mutation nature_mutation  valeur_fonciere code_commune  \
1365203    2023-01-04           Vente         160000.0        35238   
1365206    2023-01-07           Vente         320000.0        35238   
1365214    2023-01-06           Vente         120000.0        35238   
1365215    2023-01-06           Vente         120000.0        35238   
1365243    2023-01-04           Vente         310000.0        35238   
...               ...             ...              ...          ...   
1427961    2023-12-20           Vente         180000.0        35238   
1428108    2023-12-22           Vente         151480.0        35238   
1428292    2023-11-17           Vente         263000.0        35238   
1428428    2023-11-14           Vente         270000.0        35238   
1428450    2023-08-30           Vente         250000.0        35238   

        nom_commune   type_local  surface_reelle_bati  \
1365203      Rennes  Appartement                 65.0   
136

In [7]:
print(gdf_final.columns)

Index(['date_mutation', 'nature_mutation', 'valeur_fonciere', 'code_commune',
       'nom_commune', 'type_local', 'surface_reelle_bati',
       'nombre_pieces_principales', 'surface_terrain', 'longitude', 'latitude',
       'geometry', 'station_A', 'dist_metro_A', 'station_B', 'dist_metro_B',
       'prix_m2', 'ligne_proche', 'dist_min_metro'],
      dtype='object')


In [11]:
print(gdf_final.describe())

       valeur_fonciere  surface_reelle_bati  nombre_pieces_principales  \
count     3.601000e+03          3601.000000                3601.000000   
mean      2.732098e+05            61.617051                   2.911413   
std       3.061598e+05            33.450631                   1.482372   
min       1.000000e+00             1.000000                   0.000000   
25%       1.350000e+05            38.000000                   2.000000   
50%       1.980000e+05            59.000000                   3.000000   
75%       3.000000e+05            77.000000                   4.000000   
max       3.100000e+06           323.000000                  11.000000   

       surface_terrain    longitude     latitude  dist_metro_A  dist_metro_B  \
count       664.000000  3601.000000  3601.000000   3601.000000   3601.000000   
mean        231.656627    -1.675089    48.109862    986.303017    943.412193   
std         321.607289     0.018671     0.012905    728.571675    731.603302   
min          

In [ ]:
import src.stats_desc as sd

# Statistiques générales
print("--- Statistiques Globales ---")
print(sd.get_general_stats(gdf_final))

# Comparaison Ligne A vs Ligne B
print("\n--- Comparaison par Ligne de Métro ---")
stats_lignes = sd.get_stats_by_ligne(gdf_final)
display(stats_lignes)

# Statistique par tranche (distance min au métro <250, 250-500, 500-800, >800)
stats_tranches = sd.analyse_prix_dist_tranche(gdf_final)
print(stats_tranches)


--- Statistiques Globales ---
count      3601.000000
mean       4998.525626
std       11084.289842
min           0.019231
25%        3055.555556
50%        3883.548387
75%        4836.538462
max      465585.000000
Name: prix_m2, dtype: float64

--- Comparaison par Ligne de Métro ---


,mean,median,std,count
ligne_proche,,,,
A,4668.420912,3700.000000,7893.725716,1972
B,5398.136735,4103.773585,13998.322053,1629


AttributeError: module 'src.stats_desc' has no attribute 'analyse_prix_dist_tranche'

Importation des deux bases de données :
- Demandes de valeurs foncières
- Topologie des points d'arrêt de métro du réseau STAR

Points d'arrêt de métro 

In [ ]:
# URL directe vers l'export GeoJSON du portail Open Data
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

# Lecture directe depuis l'URL
gdf_metro = gpd.read_file(url_metro)

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(gdf_metro.head())
print(gdf_metro.crs)

In [ ]:
print(gdf_metro.columns)


Demandes de Valeurs Foncières (DVF)

In [ ]:
# Exemple pour l'année 2023 (ou la plus récente disponible sur l'API DVF)
# URL des fichiers consolidés par Etalab
url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

# On précise le séparateur (souvent la virgule) et on gère la compression .gz
# précisez 'low_memory=False' pour éviter les warnings sur les types
df_dvf = pd.read_csv(url_dvf, compression='gzip', sep=',', low_memory=False)

# Pour filtrer sur Rennes (code commune 35238)
df_rennes = df_dvf[df_dvf['code_commune'] == '35238'].copy()

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(df_rennes.head())

In [ ]:
print(df_rennes.columns)

Modèle de prédiction du prix :

In [ ]:
from src.model import preparer_et_entrainer

# Supposons que df_rennes et gdf_metro sont déjà chargés
model_final, features_utilisees = preparer_et_entrainer(df_rennes, gdf_metro)

print("Modèle entraîné avec succès !")
print(f"Features utilisées : {features_utilisees}")

# Vous pouvez maintenant utiliser model_final pour faire des prédictions
# exemple_data = ... 
# prediction = model_final.predict(exemple_data)

In [ ]:

from src.model import preparer_et_entrainer, predire_impact_nouvelle_station

# Exemple : un appartement de 50m², 3 pièces, en 2026
surface = 50
pieces = 3
annee = 2026
code_type = 1 # Supposons que 1 soit l'appartement

# Scénario 1 : Le métro est à 2 km (2000m)
prix_actuel = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=2000)

# Scénario 2 : Le métro est à 0 m (nouvelle station en bas de l'immeuble)
prix_avec_metro = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=0)

print(f"Prix estimé actuel : {prix_actuel:.2f} €")
print(f"Prix estimé avec nouvelle station : {prix_avec_metro:.2f} €")
print(f"Plus-value théorique : {prix_avec_metro - prix_actuel:.2f} €")